# 02 — Find the decoy *(exercise)* · *Encuentra el señuelo (ejercicio)*

### English
Your turn. Somewhere in the twelve analyzed Davis blocks is a **busy decoy**: a
street that looks alarming on a raw-count map but is not actually a high-risk
segment once you account for how many people pass through it. Using the same
statistics code the project publishes with, you will:

1. rank the segments by **raw report count** (the naive view),
2. compute the **exposure-normalized rate with a 95% confidence interval** for
   each (the honest view), and
3. find the segment that **falls the furthest** between the two rankings — that
   is the decoy.

Work through the cells and form your answer **before** opening the *Solution*
section at the bottom.

### Español
Es tu turno. En algún lugar de los doce bloques analizados de Davis hay un
**señuelo concurrido**: una calle que parece alarmante en un mapa de conteo crudo
pero que no es de alto riesgo una vez que consideras cuánta gente pasa por ella.
Con el mismo código estadístico con el que publica el proyecto, vas a:

1. ordenar los segmentos por **conteo crudo de reportes** (la vista ingenua),
2. calcular la **tasa normalizada por exposición con intervalo de confianza del
   95%** de cada uno (la vista honesta), y
3. encontrar el segmento que **más cae** entre ambas clasificaciones — ese es el
   señuelo.

Trabaja las celdas y forma tu respuesta **antes** de abrir la sección *Solución*
al final.

In [ ]:
from pathlib import Path

from nearmiss.config import load_config
from nearmiss.engine import build_analysis


def repo_root() -> Path:
    """Walk up from the kernel's working directory to the repo root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not locate the nearmiss repo root")


ROOT = repo_root()
config = load_config(ROOT / "config" / "davis-demo.toml")
bundle = build_analysis(config)

names = {s.id: s.name for s in bundle.segments}
# The twelve ANALYZED blocks — the ones that carry exposure and reports. Every
# other block in the synthetic Davis grid is published as no-exposure context.
analyzed = [s for s in bundle.result.segments if s.exposure_estimate is not None]

print("city:", config.city, "(synthetic demonstration data — not real reports)")
print("exposure coverage:", round(bundle.result.exposure_coverage * 100), "%")
print("analyzed segments:", len(analyzed))

### Step 1 — The naive view · *Paso 1 — La vista ingenua*
Rank the analyzed segments by raw report count. Which street would a
raw-count heat map paint reddest? · *Ordena los segmentos por conteo crudo.
¿Cuál pintaría de rojo más intenso un mapa de conteo?*

In [ ]:
from IPython.display import Markdown

by_count = sorted(analyzed, key=lambda s: -s.report_count)
rows = ["| Rank | Segment | Raw reports | Exposure |", "| ---: | --- | ---: | ---: |"]
for i, s in enumerate(by_count, start=1):
    rows.append(f"| {i} | {names[s.segment_id]} | {s.report_count} | {s.exposure_estimate:.0f} |")
Markdown("\n".join(rows))

> **Prompt.** Write down the top segment by raw count and *why it might be busy
> rather than dangerous* (look at its exposure column). · ***Consigna.** Anota el
> segmento más alto por conteo crudo y por qué podría estar concurrido en vez de
> ser peligroso (mira su columna de exposición).*

### Step 2 — The honest view · *Paso 2 — La vista honesta*
Now compute the exposure-normalized rate and its 95% Poisson confidence interval
for every segment, using `nearmiss.stats.rates.rate_with_ci` — the exact function
the pipeline uses. A rate is reports per `rate_per` units of exposure. One refinement: the published **primary rate** excludes records the geocoder flagged as low-confidence (`low_accuracy` / `far_snap`), so we recompute that primary numerator from the cleaned records before comparing.
· *Ahora
calcula la tasa normalizada y su intervalo de confianza de Poisson al 95% para
cada segmento, usando `nearmiss.stats.rates.rate_with_ci` — la misma función de
la tubería.* Un refinamiento: la **tasa primaria** publicada excluye los reportes que el geocodificador marcó como de baja confianza (`low_accuracy` / `far_snap`), así que recalculamos ese numerador primario a partir de los registros limpios antes de comparar.

In [ ]:
from nearmiss.stats.aggregate import _LOW_CONFIDENCE_RAW
from nearmiss.stats.rates import rate_with_ci

per = config.rate_per

# The pipeline's PRIMARY rate excludes low-confidence records (low_accuracy /
# far_snap geocodes) from the numerator; report_count stays the all-records
# total. Recompute that primary count from the cleaned records ourselves so the
# cross-check below exercises the same definition the project publishes.
primary_count = {s.segment_id: 0 for s in analyzed}
for r in bundle.records:
    if r.segment_id in primary_count and not (set(r.quality_flags) & _LOW_CONFIDENCE_RAW):
        primary_count[r.segment_id] += 1

computed = {}
for s in analyzed:
    rate, lo, hi = rate_with_ci(
        primary_count[s.segment_id], s.exposure_estimate, per, config.confidence_z
    )
    computed[s.segment_id] = (rate, lo, hi)
    # Cross-check: our hand computation matches the published pipeline output
    # (which the pipeline rounds to 4 decimals for byte-stable figures).
    assert abs(rate - s.rate) < 1e-3, (s.segment_id, rate, s.rate)

by_rate = sorted(analyzed, key=lambda s: -computed[s.segment_id][0])
rows = [f"| Rank | Segment | Rate /{int(per)} | 95% CI | n |", "| ---: | --- | ---: | --- | ---: |"]
for i, s in enumerate(by_rate, start=1):
    rate, lo, hi = computed[s.segment_id]
    rows.append(f"| {i} | {names[s.segment_id]} | {rate:.2f} | {lo:.2f}–{hi:.2f} | {s.n} |")
Markdown("\n".join(rows))

> **Prompt.** The top of *this* table is the real hotspot. Notice that the
> segment that topped the raw-count table has fallen sharply. Overlapping
> confidence intervals also warn you which "differences" are within noise. ·
> ***Consigna.** La cima de *esta* tabla es el verdadero punto caliente. El
> segmento que encabezaba la tabla de conteo crudo cayó fuertemente.*

### Step 3 — Measure the fall · *Paso 3 — Mide la caída*
Compute each segment's change in rank between the two views. The decoy is the one
that plunges the most: high on raw volume, low on rate. · *Calcula el cambio de
posición de cada segmento entre las dos vistas. El señuelo es el que más se
desploma.*

In [ ]:
count_rank = {s.segment_id: i for i, s in enumerate(by_count, start=1)}
rate_rank = {s.segment_id: i for i, s in enumerate(by_rate, start=1)}
drop = {sid: rate_rank[sid] - count_rank[sid] for sid in count_rank}

biggest = max(drop, key=lambda sid: drop[sid])
rows = ["| Segment | Count rank | Rate rank | Change |", "| --- | ---: | ---: | ---: |"]
for sid in sorted(drop, key=lambda sid: -drop[sid]):
    arrow = f"▼ {drop[sid]}" if drop[sid] > 0 else ("—" if drop[sid] == 0 else f"▲ {-drop[sid]}")
    mark = "  ← decoy" if sid == biggest else ""
    rows.append(f"| {names[sid]} | {count_rank[sid]} | {rate_rank[sid]} | {arrow}{mark} |")
Markdown("\n".join(rows))

---
## Solution · *Solución*

### English
The decoy is **`seg-03` — "3rd St (B–C)"**. It carries the **most raw reports of
any block** (12 close passes + 5 surface hazards + 3 debris = 20) yet sits near
the **bottom of the rate ranking**, because its exposure denominator (8,000 bike
trips) is an order of magnitude larger than the hotspot's. It is:

- **not** in the top three by rate,
- **not** flagged as a significant Getis-Ord Gi\* cluster, and
- exactly the segment the project's own honesty test,
  `tests/test_hotspot.py::test_busy_decoy_has_most_raw_reports_but_low_rate`,
  pins down.

The true hotspot is **`seg-06` — "5th St (C–D)"**: low exposure, a high rate, and
the centre of the only statistically significant cluster.

### Español
El señuelo es **`seg-03` — "3rd St (B–C)"**. Acumula la **mayor cantidad de
reportes crudos** (12 + 5 + 3 = 20) pero queda cerca del **fondo** en la
clasificación por tasa, porque su denominador de exposición (8,000 viajes) es un
orden de magnitud mayor que el del punto caliente. **No** está entre los tres
primeros por tasa, **no** se marca como grupo significativo Gi\*, y es justo el
segmento que fija la prueba de honestidad del proyecto en
`tests/test_hotspot.py`. El verdadero punto caliente es **`seg-06` — "5th St
(C–D)"**.

In [ ]:
decoy = bundle.result.segments  # verify the reveal against the pipeline
by_id = {s.segment_id: s for s in decoy}
seg03, seg06 = by_id["seg-03"], by_id["seg-06"]

assert biggest == "seg-03", biggest
assert seg03.report_count == max(s.report_count for s in analyzed)   # most raw reports
assert not seg03.significant                                          # but not significant
top3_by_rate = {s.segment_id for s in by_rate[:3]}
assert "seg-03" not in top3_by_rate                                  # and not near the top
assert seg06.significant and seg06.rate == max(s.rate for s in analyzed)

print("Decoy confirmed:", names["seg-03"],
      f"(raw n={seg03.report_count}, rate={seg03.rate:.2f}, significant={seg03.significant})")
print("True hotspot   :", names["seg-06"],
      f"(raw n={seg06.report_count}, rate={seg06.rate:.2f}, significant={seg06.significant})")